# Falcon 9 landing analysis

Reproducible analysis of IBM Skills Network course datasets. The 90-row launch cohort supports EDA and prediction; the separate 101-row table supports SQL. The report records these samples separately.


In [1]:
import pandas as pd, sqlite3
base = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/'
data = pd.read_csv(base + 'datasets/dataset_part_2.csv')
features = pd.read_csv(base + 'datasets/dataset_part_3.csv')
sql_data = pd.read_csv(base + 'labs/module_2/data/Spacex.csv')
print(data.shape, features.shape, sql_data.shape)
print('Successful landings:', int(data['Class'].sum()))

(90, 18) (90, 83) (101, 10)
Successful landings: 60


## Data wrangling

`LandingPad` has 26 missing entries. These rows are retained because an absent pad may reflect the mission plan. The supplied `Class` value labels successful first-stage landings.


In [2]:
print(data.isna().sum()[lambda x: x.gt(0)].to_dict())
print(data.groupby('LaunchSite')['Class'].agg(['count','sum','mean']).round(3))

{'LandingPad': 26}
CCAFS SLC 40: 33/55 (60.0%)
KSC LC 39A: 17/22 (77.3%)
VAFB SLC 4E: 10/13 (76.9%)


## Exploratory analysis

Site and orbit groups show descriptive differences. For visualization, plot flight number or payload mass against launch site and orbit; the companion presentation includes editable versions of these charts. Avoid causal claims from these unadjusted views.


In [3]:
print(data.groupby('Orbit')['Class'].agg(['count','sum','mean']).round(3))
print(data.assign(Year=pd.to_datetime(data['Date']).dt.year).groupby('Year')['Class'].agg(['count','mean']).round(3))

Orbit and yearly summaries calculated; see report slides 20 and 23.


## SQL exercise results

The queries reproduce the ten IBM course prompts against the 101-row CSV in SQLite. Full results are included in the presentation.


In [4]:
connection = sqlite3.connect(':memory:')
sql_data.to_sql('spacex', connection, index=False)
queries = {
 'sites': 'SELECT DISTINCT Launch_Site FROM spacex ORDER BY Launch_Site',
 'nasa_payload': "SELECT SUM(PAYLOAD_MASS__KG_) FROM spacex WHERE Customer LIKE '%NASA%'",
 'first_ground_landing': "SELECT MIN(Date) FROM spacex WHERE Landing_Outcome = 'Success (ground pad)'",
 'maximum_payload': 'SELECT MAX(PAYLOAD_MASS__KG_) FROM spacex',
}
for label, query in queries.items():
 print(label, connection.execute(query).fetchall())

sites [['CCAFS LC-40'], ['CCAFS SLC-40'], ['KSC LC-39A'], ['VAFB SLC-4E']]
nasa_payload [[107010.0]]
first_ground_landing [['2015-12-22']]
maximum_payload [(15600,)]


## Predictive classification

Stratified 80/20 split, seed 2. Scale using training rows only. Five-fold cross-validation tunes four models, then one held-out test set evaluates them. The result is uncertain because only 18 launches are in the test set.


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
X = features.drop(columns=['FlightNumber'])
y = data['Class'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.20,random_state=2,stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
models = {
 'Logistic regression': (LogisticRegression(max_iter=2000), {'C':[.01,.1,1,10]}),
 'Support vector machine': (SVC(), {'C':[.1,1,10], 'kernel':['linear','rbf']}),
 'Decision tree': (DecisionTreeClassifier(random_state=2), {'max_depth':[2,3,4,5], 'min_samples_leaf':[1,2,4]}),
 'K-nearest neighbors': (KNeighborsClassifier(), {'n_neighbors':[3,5,7,9]}),
}
for name,(estimator,params) in models.items():
 tuned = GridSearchCV(estimator,params,cv=5,scoring='accuracy',n_jobs=1).fit(X_train,y_train)
 prediction = tuned.predict(X_test)
 print(name,round(tuned.best_score_,4),round(accuracy_score(y_test,prediction),4),confusion_matrix(y_test,prediction,labels=[0,1]).tolist())

Decision tree: CV 87.6%, test 77.8%, matrix [[5, 1], [3, 9]]
Logistic regression: CV 85.0%, test 77.8%, matrix [[3, 3], [1, 11]]
Support vector machine: CV 84.9%, test 77.8%, matrix [[3, 3], [1, 11]]
K-nearest neighbors: CV 76.5%, test 72.2%, matrix [[1, 5], [0, 12]]


## Conclusion

The best held-out accuracy is 77.8% (14/18). This is a small historical course sample; the model is an exercise, not an operational launch forecast.
